### Notebook Overview
The purpose of this notebook:
- Generate software requirements from C code module
- The requirements are generated per function
- Compare the requirements output with ground truth for Fuze module (as provided by Stanley)
- Improve output quality with prompt engineering to match the expected requirements

### Import Libraries

In [1]:
from pycparser import c_ast, parse_file, c_generator
import re
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
from pre_processing import pre_processing, generate_functions, extract_header_comments
import warnings
warnings.filterwarnings("ignore")

### Check if CUDA is Available
Should return True

In [2]:
torch.cuda.is_available()

True

### Load CodeLLama Model

In [3]:
# ───── Load CodeLlama ─────
model_id = "codellama/CodeLlama-7b-Instruct-hf"  # or 13B if GPU has enough VRAM

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_8bit=True   # reduces GPU memory usage
)

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.0,
    do_sample=False,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Load C Code Module (Fuze.c)

In [4]:
selected_module = 'Fuze.c'
Git_folder = "P3_MCP_Application/cmake-src/src"
with open(Git_folder+"/"+selected_module, "r") as f:
    source_code = f.read()

**Extract all comments with their line positions**

In [5]:
code_lines = source_code.splitlines()
comments = re.findall(r'//.*|/\*[\s\S]*?\*/', source_code, flags=re.MULTILINE)

# Extract line numbers (optional) to map to AST nodes
comment_positions = []
for comment in comments:
    # Find line numbers where this comment appears
    for i, line in enumerate(code_lines):
        if comment in line:
            comment_positions.append((i+1, comment))  # 1-based line number
            break
comment_positions

[(82, '/* FMU152 valid delays[ms]: */'),
 (83, '/* 0, 5, 15, 25, 35, 45, 60, 90, 180, 240 */'),
 (127,
  '/* Fuze settings can only be sent once it is confirmed that the fuze initialization is complete, and that the fuze can accept new settings */'),
 (130, '/* TRUE */'),
 (131, '/* FALSE */'),
 (133,
  '/* The FMU does not support a fuze delay of 10ms, therefore the delay is set at 15ms */'),
 (130, '/* TRUE */'),
 (192, '/* Update fuze status if fuze initialization is complete */'),
 (130, '/* TRUE */'),
 (195, '/* Update fuze status if settings are valid */'),
 (214,
  '/* Retransmit the fuze settings to the IF as the previous command was not received */'),
 (131, '/* FALSE */'),
 (241, '/* Fuze settings update successful */'),
 (130, '/* TRUE */'),
 (253, '/* Fuze settings update unsuccessful */'),
 (131, '/* FALSE */'),
 (130, '/* TRUE */'),
 (298, '/* No action needed for other cases */')]

**Extract all functions in the module with their start and end positions**

In [6]:
output_file, input_file = pre_processing(selected_module)
functions = generate_functions(output_file, input_file)
header_comments = extract_header_comments(input_file)

function FuzeInit at line 40:66
function FuzeStatusMsgSendRequest at line 70:76
function FuzeFMU152_CheckDelay at line 81:116
function Fuze_SetDelay at line 119:162
function Fuze_HandleStatusMsg at line 165:223
function FuzeHandleRxData at line 228:264
function Fuze_Service at line 268:297


**Function to extract global variables and embed them in function**

In [7]:
class ContextExtractor(c_ast.NodeVisitor):
    def __init__(self):
        self.typedefs = {}
        self.enums = {}
        self.structs = {}
        self.globals = {}
        self.functions = {}
        self.generator = c_generator.CGenerator()

    def visit_Typedef(self, node):
        name = node.name
        self.typedefs[name] = self.generator.visit(node)

    def visit_Enum(self, node):
        name = node.name or "<anonymous_enum>"
        self.enums[name] = self.generator.visit(node)

    def visit_Struct(self, node):
        name = node.name or "<anonymous_struct>"
        self.structs[name] = self.generator.visit(node)

    def visit_Decl(self, node):
        # Capture global variables (not inside a function)
        if isinstance(node.type, (c_ast.TypeDecl, c_ast.PtrDecl)):
            if not hasattr(node, "funcspec") or not node.funcspec:
                if not isinstance(node.type.type, c_ast.FuncDecl):  # not a function
                    name = node.name
                    self.globals[name] = self.generator.visit(node)

    def visit_FuncDef(self, node):
        # Store function definitions (signatures only)
        func_name = node.decl.name
        self.functions[func_name] = node

    def visit_FuncDecl(self, node):
        # Store function declarations (prototypes)
        if hasattr(node.type, 'declname'):
            func_name = node.type.declname
            self.functions[func_name] = node

ast = parse_file(output_file)
extractor = ContextExtractor()
extractor.visit(ast)
generator = c_generator.CGenerator()

**Function to embed comments inside the function**

In [8]:
def add_comments(func_code):
    func_code_comments = []
    func_code_startline = functions[i]['start line']

    code_line = func_code_startline
    for line in func_code.split('\n'):
        func_code_comments.append(line)
        code_line+=1
        if code_line <= functions[i]['end line']:
            for pos in comment_positions:
                if pos[0] == code_line:
                    func_code_comments.append(pos[1])
                    code_line+=1

    return '\n'.join(func_code_comments)

### Define LLM Prompt

In [9]:
reqs_prompt_template = """
You are an **expert system engineer** specialized in writing high-level software requirements from implementation code.
Treat the provided function implementation as the sole source of truth and produce only natural-language requirements.

### Inputs (informational)
- function_name: the implementation name (do not use this in outputs).
- function_code: the implementation to analyze.

### Task
Analyze the provided function implementation and produce a **comprehensive, numbered list of high-level software requirements** that reflect all observable behaviors encoded in the implementation.
Each logical condition (e.g., `if`, `elif`, `else`, `switch`, `try/except`, `return`, `raise`, etc.) must be translated into one or more distinct requirements describing the system’s intended behavior for that condition or branch.
If the function contains multiple cases (e.g., `switch/case` statements or multiple `if/elif` branches), generate **separate, detailed requirements for each case**. Do not merge multiple behaviors into a single requirement.

Focus on:
- Functional requirements (what the module does)
- Interface requirements (data structures, APIs)
- Performance requirements (if timing/resource constraints in code).
- Safety/Security requirements (if applicable)

Write the requirements as **SHALL statements**. Individual sentences do not all need to begin with "The system shall" or similar prefixes.

### Output Format (strict — exact, no exceptions)
1. Output only a numbered list (1., 2., 3., ...), one requirement per line.
2. Each requirement must be exactly one sentence in plain natural language.
3. Do not include any additional text, headings, examples, code, or commentary.
4. When expressing numeric values, preserve the exact digits as written in the source code.

### Mandatory Rules (strict — no exceptions)
- Do NOT mention or copy any identifiers from the code (function names, variable names, parameter names, type names).
- Do NOT use code keywords, syntax, or code-like fragments (for example: if, while, ==, //, ->).
- Do NOT use phrases such as "the function does", "this function", "callers", or similar implementation-level wording.
- Do NOT describe algorithmic steps, internal loops, or implementation strategies; describe only observable behavior and obligations.
- Do not use **code-specific terminology** such as bits, flags, registers, memory, pointers, or low-level implementation details.
- Every requirement must be directly supported by behavior or constants present in the provided code; do not invent capabilities or assumptions not present in the code.
- Requirements must be unique and non-overlapping.
- When expressing numeric values, preserve the exact digits as written in the source code.

### What to describe (priority order)
- Observable functionality and outcomes.
- Inputs and outputs in conceptual terms (types, formats, ranges) — without variable names.
- Constraints, limits, thresholds, timeouts, and formats (express numeric values exactly as in the code).
- Error handling and failure behavior (what observable error or response occurs).

### If nothing determinable
If no requirements can be reasonably inferred from the code, output exactly one item:
1. There are no determinable requirements based on the provided code.

### Provided data
Function Name: {function_name}
Function Code:
{function_code}

### Requirements:
"""


### Test LLM for each Function in Module

**Function 1: FuzeInit**

In [10]:
i = 0

## Extract function snippet with its comments
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

## Update prompt template
reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
header_comments = header_comments
)

## View prompt
print(reqs_prompt)


You are an **expert system engineer** specialized in writing high-level software requirements from implementation code.
Treat the provided function implementation as the sole source of truth and produce only natural-language requirements.

### Inputs (informational)
- function_name: the implementation name (do not use this in outputs).
- function_code: the implementation to analyze.

### Task
Analyze the provided function implementation and produce a **comprehensive, numbered list of high-level software requirements** that reflect all observable behaviors encoded in the implementation.
Each logical condition (e.g., `if`, `elif`, `else`, `switch`, `try/except`, `return`, `raise`, etc.) must be translated into one or more distinct requirements describing the system’s intended behavior for that condition or branch.
If the function contains multiple cases (e.g., `switch/case` statements or multiple `if/elif` branches), generate **separate, detailed requirements for each case**. Do not mer

In [11]:
## Run LLM and View Response

response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall initialize the fuze parameters when the aux power is turned on.
2. The system shall set the initial delay command count to zero.
3. The system shall request status transmission every 5 seconds until it receives a successful response.
4. The system shall send a set delay command after receiving a successful response.


**Function 2: FuzeStatusMsgSendRequest**

In [12]:
i = 1

## Extract function snippet with its comments
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

## Update prompt template
reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
)

## Run LLM and View Response
response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall send a request status command message over the CAN bus when requested.
2. The system shall increment the fuze request status transmission counter every time it sends a request status command message.
3. The system shall log the current timestamp and the value of the fuze request status transmission counter whenever it sends a request status command message.


In [22]:
print(function_code_withcomments)

void FuzeStatusMsgSendRequest(void)
{
  IF_SendMsg((uint8_t) 0x21, 0, 0);
  SS_FUZE.Fuze.RequestStatusTxCnt++;
  if (AUXCTRL_SS.FuzeControls.TxAuxOn != 0U)
  {
    AUXsprintf("%u:FUZE Request Status Command Sent %d\n", GetTimeUs(), SS_FUZE.Fuze.RequestStatusTxCnt);
  }
}




**Function 3: FuzeFMU152_CheckDelay**

In [13]:
i = 2
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
)

response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall accept input parameters ranging between 0 and 240 milliseconds.
2. The system shall filter input parameters within the range [0, 3] milliseconds to return value 0.
3. The system shall filter input parameters within the range [3, 10] milliseconds to return value 5.
4. The system shall filter input parameters within the range [10, 20] milliseconds to return value 15.
5. The system shall filter input parameters within the range [20, 30] milliseconds to return value 25.
6. The system shall filter input parameters within the range [30, 40] milliseconds to return value 35.
7. The system shall filter input parameters within the range [40, 50] milliseconds to return value 45.
8. The system shall filter input parameters within the range [50, 75] milliseconds to return value 60.
9. The system shall filter input parameters within the range [75, 135] milliseconds to return value 90.
10. The system shall filter input parameters within the range [135, 210] milliseconds to return

**Function 4: Fuze_SetDelay**

In [14]:
i = 3
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
)

response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall allow setting the delay between sending commands to the fuze.
2. The system shall send a command to update the fuze's delay when receiving a valid request from a node.
3. The system shall respond with success status after updating the fuze's delay.
4. The system shall ignore requests to change the fuze's delay if the fuze has already been initialized.
5. The system shall ignore requests to change the fuze's delay if the fuze cannot receive new settings.


**Function 5: Fuze_HandleStatusMsg**

In [15]:
i = 4
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
)

response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall receive a FUZE status message.
2. The system shall update its software version when receiving a FUZE status message.
3. The system shall update its status when receiving a FUZE status message.
4. The system shall increment its set delay count when receiving a FUZE status message.
5. The system shall print the following information when receiving a FUZE status message:
	* Time stamp
	* Number of times it has received a FUZE status message
	* Software version
	* Status
	* Initialization complete flag
	* Communication OK flag
	* Updating delay flag
	* Fuze 5 V power supply ON flag
	* Fuze 15 V power supply ON flag
	* Fuze 24 V power supply ON flag
	* Fuze arm ON flag
	* Fuze ID
	* Spare
6. The system shall update its type when receiving a FUZE status message.
7. The system shall update its mode when receiving a FUZE status message.
8. The system shall update its time delay when receiving a FUZE status message.
9. The system shall build a telemetry block every two seco

**Function 6: FuzeHandleRxData**

In [16]:
i = 5
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
)

response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall receive incoming messages addressed to it.
2. The system shall process the message contents according to its protocol specification.
3. The system shall send acknowledgment messages when appropriate.



**Function 7: Fuze_Service**

In [18]:
i = 6
function_code_parsed = generator.visit(extractor.functions[functions[i]['name']])
function_code_withcomments = add_comments(function_code_parsed)

reqs_prompt = reqs_prompt_template.format(
function_name = functions[i]['name'],
function_code = function_code_withcomments,
)

response = llm(reqs_prompt)
print(response.split('### Requirements:')[1])


1. The system shall update its time stamp when it receives a fuze card fitted signal.
2. The system shall calculate the difference between two consecutive time stamps.
3. The system shall send a service control command to the fuze controller every 5 seconds if the difference between two consecutive time stamps exceeds 200 milliseconds.
4. The system shall set the old time stamp equal to the current time stamp if the difference between two consecutive time stamps exceeds 200 milliseconds.
5. The system shall initialize the fuze mode to zero if the initialization complete flag is false.
6. The system shall send a service control command to the fuze controller if the updating flag is true.
7. The system shall increment the sending count of the service control command to the fuze controller if the updating flag is true.
